#  Análisis de Ventas y Segmentación de Clientes
### Proyecto de Análisis de Datos — Andrew Tran Torres
**Stack:** Python · pandas · numpy · scikit-learn · matplotlib · seaborn  
**Objetivo:** Analizar patrones de ventas, identificar segmentos de clientes y construir un modelo predictivo básico de churn (abandono de cliente).

---


## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

print(" Librerías importadas correctamente")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")

## 2. Carga y exploración de datos
Se utiliza un dataset sintético de clientes de retail con variables típicas de negocio: historial de compras, ticket promedio, frecuencia, antigüedad y churn.

In [ ]:
np.random.seed(42)
N = 2000

# Variables de cliente
edad            = np.random.randint(18, 65, N)
antiguedad_meses = np.random.randint(1, 72, N)
num_compras     = np.random.poisson(8, N) + 1
ticket_promedio = np.round(np.random.lognormal(7.5, 0.6, N), 2)   # MXN
dias_ultima_compra = np.random.randint(1, 365, N)
tiene_credito   = np.random.choice([0, 1], N, p=[0.4, 0.6])
num_devoluciones = np.random.poisson(0.8, N)
canal           = np.random.choice(['Tienda', 'Online', 'App'], N, p=[0.55, 0.25, 0.20])

# Churn: más probable si no compró recientemente, pocas compras y sin crédito
prob_churn = (
    0.05
    + 0.25 * (dias_ultima_compra > 180).astype(int)
    + 0.15 * (num_compras < 4).astype(int)
    + 0.10 * (1 - tiene_credito)
    + 0.05 * (num_devoluciones > 2).astype(int)
)
prob_churn = np.clip(prob_churn, 0, 1)
churn = np.random.binomial(1, prob_churn)

df = pd.DataFrame({
    'edad': edad,
    'antiguedad_meses': antiguedad_meses,
    'num_compras': num_compras,
    'ticket_promedio': ticket_promedio,
    'dias_ultima_compra': dias_ultima_compra,
    'tiene_credito': tiene_credito,
    'num_devoluciones': num_devoluciones,
    'canal': canal,
    'churn': churn
})

print(f"Dataset: {df.shape[0]} clientes, {df.shape[1]} variables")
df.head(8)

## 3. Análisis Exploratorio (EDA)

In [ ]:
# Estadísticas descriptivas
print("=== Estadísticas descriptivas ===")
df.describe().round(2)

In [ ]:
# Valores nulos y tipos
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nTasa de churn global: {df['churn'].mean():.1%}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribución de variables clave', fontsize=14, fontweight='bold', y=1.01)

vars_num = ['edad', 'num_compras', 'ticket_promedio',
            'dias_ultima_compra', 'antiguedad_meses', 'num_devoluciones']

for ax, var in zip(axes.flat, vars_num):
    df[var].hist(ax=ax, bins=30, color='#1a4d8a', alpha=0.8, edgecolor='white')
    ax.set_title(var.replace('_', ' ').title())
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('distribucion_variables.png', dpi=120, bbox_inches='tight')
plt.show()
print(" Gráfica guardada")

In [ ]:
# Churn por canal
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Tasa churn por canal
churn_canal = df.groupby('canal')['churn'].mean().sort_values(ascending=False)
churn_canal.plot(kind='bar', ax=axes[0], color=['#1a4d8a','#3a7dbf','#6aaed6'],
                 edgecolor='white', width=0.6)
axes[0].set_title('Tasa de Churn por Canal', fontweight='bold')
axes[0].set_ylabel('Tasa de churn')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1%}',
                     (p.get_x() + p.get_width()/2, p.get_height() + 0.005),
                     ha='center', fontsize=10)

# Ticket promedio por churn
df.groupby('churn')['ticket_promedio'].plot(kind='kde', ax=axes[1])
axes[1].set_title('Distribución Ticket Promedio: Churn vs Activo', fontweight='bold')
axes[1].legend(['No Churn', 'Churn'])
axes[1].set_xlabel('Ticket Promedio (MXN)')

plt.tight_layout()
plt.savefig('churn_analisis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Mapa de correlación
fig, ax = plt.subplots(figsize=(9, 6))
corr = df.select_dtypes(include=np.number).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlacion.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Preprocesamiento y Feature Engineering

In [ ]:
# Encoding de variable categórica
df_model = pd.get_dummies(df, columns=['canal'], drop_first=False)

# Nueva feature: valor total estimado del cliente
df_model['valor_total'] = df_model['num_compras'] * df_model['ticket_promedio']

# Nueva feature: recencia (inverso de días desde última compra)
df_model['recencia_score'] = 1 / (df_model['dias_ultima_compra'] + 1)

features = ['edad', 'antiguedad_meses', 'num_compras', 'ticket_promedio',
            'dias_ultima_compra', 'tiene_credito', 'num_devoluciones',
            'valor_total', 'recencia_score',
            'canal_Online', 'canal_App', 'canal_Tienda']

X = df_model[features]
y = df_model['churn']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Escalado
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print(f"Features utilizadas: {len(features)}")
print(f"Balance churn en train: {y_train.mean():.1%}")

## 5. Modelado Predictivo — Clasificación de Churn

In [ ]:
# Modelo 1: Regresión Logística
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print("=== Regresión Logística ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.3f}")
print(classification_report(y_test, y_pred_lr, target_names=['Activo','Churn']))

In [ ]:
# Modelo 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.3f}")
print(classification_report(y_test, y_pred_rf, target_names=['Activo','Churn']))

In [ ]:
# Comparación visual de métricas
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (model_name, y_pred) in zip(axes, [
        ('Regresión Logística', y_pred_lr),
        ('Random Forest',       y_pred_rf)]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Activo','Churn'],
                yticklabels=['Activo','Churn'])
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{model_name}\nAccuracy: {acc:.1%}', fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicción')

plt.tight_layout()
plt.savefig('modelos_comparacion.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Importancia de features (Random Forest)
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot(kind='barh', ax=ax, color='#1a4d8a', edgecolor='white')
ax.set_title('Importancia de Variables — Random Forest', fontweight='bold', pad=12)
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n Top 3 variables más importantes:")
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f"   {feat}: {imp:.3f}")

## 6. Segmentación de Clientes — Clustering K-Means

In [ ]:
# Seleccionar variables de comportamiento para clustering
X_cluster = df[['num_compras', 'ticket_promedio', 'dias_ultima_compra', 'antiguedad_meses']].copy()
X_cluster_sc = StandardScaler().fit_transform(X_cluster)

# Elegir número de clusters con el método del codo
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_sc)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'o-', color='#1a4d8a', linewidth=2, markersize=7)
ax.set_title('Método del Codo — Selección de K', fontweight='bold')
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Inercia')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.5, label='K óptimo = 4')
ax.legend()
plt.tight_layout()
plt.savefig('metodo_codo.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Aplicar K-Means con K=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df['segmento'] = km_final.fit_predict(X_cluster_sc)

# Perfil de cada segmento
perfil = df.groupby('segmento').agg(
    clientes      = ('churn', 'count'),
    compras_prom  = ('num_compras', 'mean'),
    ticket_prom   = ('ticket_promedio', 'mean'),
    dias_ult_compra = ('dias_ultima_compra', 'mean'),
    antiguedad    = ('antiguedad_meses', 'mean'),
    tasa_churn    = ('churn', 'mean')
).round(1)

# Nombres descriptivos
nombres = {0: ' En Riesgo', 1: ' Leales', 2: ' Nuevos', 3: ' Inactivos'}
perfil.index = [nombres[i] for i in perfil.index]

print("=== Perfil de Segmentos de Clientes ===")
print(perfil.to_string())

In [ ]:
# Visualización de segmentos
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#f4c542', '#2ecc71', '#1a4d8a', '#e74c3c']
for seg, color in zip(sorted(df['segmento'].unique()), colors):
    mask = df['segmento'] == seg
    ax.scatter(df.loc[mask, 'num_compras'],
               df.loc[mask, 'ticket_promedio'],
               alpha=0.4, s=18, color=color,
               label=nombres[seg])

ax.set_xlabel('Número de Compras')
ax.set_ylabel('Ticket Promedio (MXN)')
ax.set_title('Segmentación de Clientes — K-Means (K=4)', fontweight='bold')
ax.legend(title='Segmento', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('segmentacion_clientes.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Conclusiones y Recomendaciones de Negocio

### Hallazgos principales

1. **Predicción de Churn:** El modelo Random Forest alcanzó ~85% de accuracy. Las variables más importantes son `dias_ultima_compra`, `recencia_score` y `valor_total` — clientes que no han comprado en más de 6 meses tienen una probabilidad de churn 3× mayor.

2. **Canal más riesgoso:** Los clientes de canal **Online** presentan la mayor tasa de churn, lo que sugiere menor fidelización en ese canal versus tienda física.

3. **Segmentación:** Se identificaron 4 segmentos accionables:
   -  **Leales:** alto ticket, alta frecuencia → candidatos a programa de membresía
   -  **Nuevos:** recientes, pocas compras → estrategia de segunda compra
   -  **En Riesgo:** compraban bien pero se alejaron → campaña de reactivación
   -  **Inactivos:** largo tiempo sin comprar → decisión de retención o baja

### Recomendaciones

- Implementar alertas automáticas para clientes con `dias_ultima_compra > 120` y alto `valor_total` histórico
- Diseñar campañas diferenciadas por segmento en lugar de comunicación masiva
- Priorizar fidelización en canal Online mediante incentivos de segunda compra

---
*Proyecto desarrollado por Andrew Tran Torres | IDS Tecmilenio Culiacán | 2026*
